# Track 1 Smart Campus - prediction notebook (c13_avg_c11_007)

Inference only: loads the trained artifact `model.pkl` (trained locally, see the training notebook) and predicts `energy_usage` for every row of the input file, in input order. All features are row-local, so shuffled rows / a missing `id` column are fine.

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

TEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")

In [ ]:
# Feature engineering - identical to training (src/features.py)
import numpy as np
import pandas as pd

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
BUILDINGS = ["ADM_A", "BUS_A", "BUS_B", "ENG_A", "ENG_B", "LEC_A", "LIB_A",
             "RES_A", "RES_B", "SCI_A", "SCI_B", "SPT_A"]
TYPES = ["Administration", "Business", "Engineering", "LectureHall", "Library",
         "Residential", "Science", "Sports"]
BUILDING_TYPE = {"ADM_A": "Administration", "BUS_A": "Business", "BUS_B": "Business",
                 "ENG_A": "Engineering", "ENG_B": "Engineering", "LEC_A": "LectureHall",
                 "LIB_A": "Library", "RES_A": "Residential", "RES_B": "Residential",
                 "SCI_A": "Science", "SCI_B": "Science", "SPT_A": "Sports"}
NUMERIC = ["previous_usage", "occupancy", "temperature", "humidity"]
# physically plausible ranges; values outside are treated as sensor errors -> missing
VALID = {"previous_usage": (0, 1000), "occupancy": (0, 10000), "temperature": (-20, 60), "humidity": (0, 100)}


def _num(s):
    return pd.to_numeric(s, errors="coerce").astype(float)


def clean_input(df):
    """Sanitise raw columns; anything unusable becomes NaN (handled downstream)."""
    out = pd.DataFrame(index=df.index)
    get = lambda c: df[c] if c in df.columns else pd.Series(np.nan, index=df.index)
    out["building_id"] = get("building_id").astype(str).str.strip().str.upper()
    out["building_type"] = get("building_type").astype(str).str.strip()
    hour = _num(get("hour"))
    out["hour"] = hour.where(hour.between(0, 23)).round()
    month = _num(get("month"))
    out["month"] = month.where(month.between(1, 12)).round()
    day = get("day_of_week").astype(str).str.strip().str[:3].str.lower()
    out["dow"] = day.map({d[:3].lower(): i for i, d in enumerate(DAYS)}).astype(float)
    for c in NUMERIC:
        v = _num(get(c))
        lo, hi = VALID[c]
        out[c] = v.where(v.between(lo, hi))
    return out


def make_features(df):
    """Tabular features for tree models (used by the GBM baseline)."""
    c = clean_input(df)
    out = pd.DataFrame(index=df.index)
    out["building"] = pd.Categorical(c["building_id"], categories=BUILDINGS)
    out["btype"] = pd.Categorical(c["building_type"], categories=TYPES)
    out["hour"] = c["hour"]
    out["dow"] = c["dow"]
    out["weekend"] = (c["dow"] >= 5).astype(float).where(c["dow"].notna())
    out["month"] = c["month"]
    for col in ["temperature", "humidity", "occupancy", "previous_usage"]:
        out[col] = c[col]
    return out


def _soft_building(c):
    """n x 12 building weights: one-hot if known, else uniform over buildings of the given type,
    else uniform over all buildings (=> prediction is the average building's expectation)."""
    n = len(c)
    W = np.zeros((n, len(BUILDINGS)))
    bid = c["building_id"].to_numpy(); bt = c["building_type"].to_numpy()
    for i in range(n):
        if bid[i] in BUILDINGS:
            W[i, BUILDINGS.index(bid[i])] = 1.0
        else:
            same = [j for j, b in enumerate(BUILDINGS) if BUILDING_TYPE[b] == bt[i]]
            W[i, same if same else slice(None)] = 1.0 / (len(same) if same else len(BUILDINGS))
    return W


def _soft_onehot(v, k):
    v = np.asarray(v, dtype=float)
    out = np.full((len(v), k), 1.0 / k)
    ok = ~np.isnan(v)
    out[ok] = 0.0
    out[np.where(ok)[0], v[ok].astype(int)] = 1.0
    return out


DEFAULT_SPEC = {"hours": "type", "weekend": "building",
                "slopes": {"previous_usage": "shared", "occupancy": "building",
                           "temperature": "building", "humidity": "shared"},
                "temp2": True, "season": True}


def linear_design(c, drop=(), spec=DEFAULT_SPEC):
    """Design matrix from a clean_input() frame. `drop` = numeric columns unavailable for this
    missingness-pattern sub-model. Categorical unknowns enter as expectations (soft one-hots)."""
    if spec.get("missing_spec") and set(drop) & {"occupancy", "previous_usage"}:
        # sub-models without occupancy/previous_usage must rebuild them from time x building -> richer design
        spec = {**spec, **spec["missing_spec"]}
    n = len(c)
    B = _soft_building(c)
    T = B @ np.array([[BUILDING_TYPE[b] == t for t in TYPES] for b in BUILDINGS], dtype=float)
    G = {"building": B, "type": T, "shared": np.ones((n, 1))}
    K = spec.get("hour_fourier", 0)
    if K:  # smooth daily profile: K harmonics (missing hour -> 0 = daily average)
        hh = c["hour"].to_numpy()
        H = np.concatenate([f(2 * np.pi * k * hh / 24)[:, None] for k in range(1, K + 1) for f in (np.sin, np.cos)], 1)
        H = np.nan_to_num(H, nan=0.0)
    else:
        H = _soft_onehot(c["hour"], 24)
    dow = c["dow"].to_numpy()
    we = np.where(np.isnan(dow), 2 / 7, (dow >= 5).astype(float))[:, None]
    cols = [B, (G[spec["hours"]][:, :, None] * H[:, None, :]).reshape(n, -1), G[spec["weekend"]] * we]
    if spec.get("hour_dev_scale"):  # per-hour deviations from the smooth profile, shrunk by a heavier
        # effective ridge penalty (alpha / scale^2) -> captures schedule steps (e.g. residential 08:00 / 21:00)
        Hd = _soft_onehot(c["hour"], 24) * spec["hour_dev_scale"]
        Gd = G[spec.get("hour_dev_group", "type")]
        cols.append((Gd[:, :, None] * Hd[:, None, :]).reshape(n, -1))
    if spec.get("weekend_hours"):  # separate daily shape on weekends
        Gw = G[spec["weekend_hours"]]
        cols.append((Gw[:, :, None] * (H * we)[:, None, :]).reshape(n, -1))
    for col in NUMERIC:
        if col not in drop:
            cols.append(G[spec["slopes"][col]] * c[col].to_numpy()[:, None])
    # extra shrinkable interactions: (numeric col, group, scale); effective penalty alpha/scale^2
    if spec.get("interactions"):
        hr = c["hour"].to_numpy()
        bands = np.stack([(hr <= 6), (hr >= 7) & (hr <= 11), (hr >= 12) & (hr <= 17), (hr >= 18)], 1).astype(float)
        bands[np.isnan(hr)] = 0.25
        we2 = np.concatenate([1 - we, we], 1)
        GX = {**G, "hourband": bands, "weekend": we2}
        for col, grp, scale in spec["interactions"]:
            if col in drop:
                continue
            if col == "temp_hinge":
                v = np.stack([np.maximum(c["temperature"].to_numpy() - k, 0) for k in (26.0, 28.0, 30.0)], 1)
                if "temperature" in drop:
                    continue
                cols.append((GX[grp][:, :, None] * v[:, None, :]).reshape(n, -1) * scale)
                continue
            cols.append(GX[grp] * c[col].to_numpy()[:, None] * scale)
    if spec.get("res_steps"):  # explicit residential day (08-21) / night schedule step indicators
        h = c["hour"].to_numpy(); res = T[:, TYPES.index("Residential")]
        day = np.where(np.isfinite(h), ((h >= 8) & (h < 21)).astype(float), 13 / 24)
        cols.append(np.stack([day * res, (1 - day) * res], 1))
    if spec.get("season", True):
        m = c["month"].to_numpy()
        sc = np.stack([np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)], 1)
        cols.append(np.nan_to_num(sc, nan=0.0))
    if spec.get("temp2", True) and "temperature" not in drop:
        cols.append(((c["temperature"].to_numpy() - 28.0) ** 2)[:, None])
    return np.concatenate(cols, 1)


def pattern_keys(c):
    miss = c[NUMERIC].isna().to_numpy()
    return [tuple(col for col, mm in zip(NUMERIC, row) if mm) for row in miss]


def predict_linear(art, df):
    """Pattern-submodel linear prediction: each row uses the model fitted without its missing
    numeric columns. art = {"coefs": {pattern: (coef, intercept)}, "spec": ..., "clip": (lo, hi)}."""
    c = clean_input(df)
    keys = pattern_keys(c)
    preds = np.empty(len(c))
    for k in set(keys):
        idx = np.array([kk == k for kk in keys])
        coef, icpt = art["coefs"][k]
        preds[idx] = linear_design(c[idx], drop=set(k), spec=art["spec"]) @ coef + icpt
    return np.clip(preds, *art["clip"])

In [ ]:
artifact = joblib.load("model.pkl")  # uploaded next to this notebook
test = pd.read_csv(TEST_PATH)
# --- component 1: c11 (our pipeline)
_both = artifact
artifact = _both['c11']

def _dev_keys(F, level):
    cols = [F[x].cat.codes.to_numpy() if hasattr(F[x], "cat") else F[x].to_numpy() for x in level]
    return list(zip(*cols))

def add_dev(F, tables, levels, skip):
    """Append deviation features (fixed order: level, then column) - identical in training and inference."""
    G = F.copy()
    for i, level in enumerate(levels):
        keys = _dev_keys(F, level)
        for c in ["occupancy", "previous_usage", "temperature", "humidity"]:
            if c != skip:
                mu = np.array([tables[i][c].get(k, np.nan) for k in keys], dtype=float)
                G[f"dev{i}_{c}"] = F[c].to_numpy(dtype=float) - mu
    return G

def add_cyclic(F):
    """sin/cos of hour and month for the imputers (appended after 'dom')."""
    G = F.copy(); h = G["hour"].to_numpy(dtype=float); m = G["month"].to_numpy(dtype=float)
    G["h_sin"], G["h_cos"] = np.sin(2 * np.pi * h / 24), np.cos(2 * np.pi * h / 24)
    G["m_sin"], G["m_cos"] = np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)
    return G

import lightgbm as lgb
# 1) fill missing numeric inputs: average of 3 target-free LightGBM imputers (domain flag = test distribution),
#    with deviation features (observed value minus its building/hour profile mean) and optional cyclic time features
F = make_features(test)
F["dom"] = artifact["domain"]
if artifact["cyclic"]:
    F = add_cyclic(F)
X = test.copy()
for col in artifact["impute_cols"]:
    miss = F[col].isna().to_numpy()
    if miss.any():
        Xm = add_dev(F, artifact["dev_tables"], artifact["dev_levels"], col).drop(columns=[col])[miss]
        X[col] = F[col].to_numpy()
        X.loc[miss, col] = np.mean([lgb.Booster(model_str=s).predict(Xm) for s in artifact["imputers"][col]], axis=0)
# 2) linear energy model (extrapolates linearly in every observed feature)
preds = predict_linear(artifact["linear"], X)
# 3) optional offset for rows whose previous_usage was missing (estimated on naturally-missing train rows)
preds = preds + artifact["prev_missing_offset"] * F["previous_usage"].isna().to_numpy()
_p11 = np.asarray(preds, dtype=float)
# --- component 2: submission 007, run in its own namespace with its original inference code
_F007_SRC = 'import os\nos.environ.setdefault("OMP_NUM_THREADS", "4")\nos.environ.setdefault("OPENBLAS_NUM_THREADS", "4")\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport lightgbm as lgb\nfrom threadpoolctl import threadpool_limits\nTEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")\nOUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")\n\n"""Row-local feature engineering + linear-model inference. This exact code is copied into the\nprediction notebook, so it may only use numpy / pandas.\n\nRobustness contract (the private test "may contain different observations and edge cases"):\nevery row gets a finite prediction even with missing / malformed / unseen values in ANY column.\n"""\nimport numpy as np\nimport pandas as pd\n\nDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]\nBUILDINGS = ["ADM_A", "BUS_A", "BUS_B", "ENG_A", "ENG_B", "LEC_A", "LIB_A",\n             "RES_A", "RES_B", "SCI_A", "SCI_B", "SPT_A"]\nTYPES = ["Administration", "Business", "Engineering", "LectureHall", "Library",\n         "Residential", "Science", "Sports"]\nBUILDING_TYPE = {"ADM_A": "Administration", "BUS_A": "Business", "BUS_B": "Business",\n                 "ENG_A": "Engineering", "ENG_B": "Engineering", "LEC_A": "LectureHall",\n                 "LIB_A": "Library", "RES_A": "Residential", "RES_B": "Residential",\n                 "SCI_A": "Science", "SCI_B": "Science", "SPT_A": "Sports"}\nNUMERIC = ["previous_usage", "occupancy", "temperature", "humidity"]\n# physically plausible ranges; values outside are treated as sensor errors -> missing\nVALID = {"previous_usage": (0, 1000), "occupancy": (0, 10000), "temperature": (-20, 60), "humidity": (0, 100)}\n\n\ndef _num(s):\n    return pd.to_numeric(s, errors="coerce").astype(float)\n\n\ndef clean_input(df):\n    """Sanitise raw columns; anything unusable becomes NaN (handled downstream)."""\n    out = pd.DataFrame(index=df.index)\n    get = lambda c: df[c] if c in df.columns else pd.Series(np.nan, index=df.index)\n    out["building_id"] = get("building_id").astype(str).str.strip().str.upper()\n    out["building_type"] = get("building_type").astype(str).str.strip()\n    hour = _num(get("hour"))\n    out["hour"] = hour.where(hour.between(0, 23)).round()\n    month = _num(get("month"))\n    out["month"] = month.where(month.between(1, 12)).round()\n    day = get("day_of_week").astype(str).str.strip().str[:3].str.lower()\n    out["dow"] = day.map({d[:3].lower(): i for i, d in enumerate(DAYS)}).astype(float)\n    for c in NUMERIC:\n        v = _num(get(c))\n        lo, hi = VALID[c]\n        out[c] = v.where(v.between(lo, hi))\n    return out\n\n\ndef make_features(df):\n    """Tabular features for tree models (used by the GBM baseline)."""\n    c = clean_input(df)\n    out = pd.DataFrame(index=df.index)\n    out["building"] = pd.Categorical(c["building_id"], categories=BUILDINGS)\n    out["btype"] = pd.Categorical(c["building_type"], categories=TYPES)\n    out["hour"] = c["hour"]\n    out["dow"] = c["dow"]\n    out["weekend"] = (c["dow"] >= 5).astype(float).where(c["dow"].notna())\n    out["month"] = c["month"]\n    for col in ["temperature", "humidity", "occupancy", "previous_usage"]:\n        out[col] = c[col]\n    return out\n\n\ndef _soft_building(c):\n    """n x 12 building weights: one-hot if known, else uniform over buildings of the given type,\n    else uniform over all buildings (=> prediction is the average building\'s expectation)."""\n    n = len(c)\n    W = np.zeros((n, len(BUILDINGS)))\n    bid = c["building_id"].to_numpy(); bt = c["building_type"].to_numpy()\n    for i in range(n):\n        if bid[i] in BUILDINGS:\n            W[i, BUILDINGS.index(bid[i])] = 1.0\n        else:\n            same = [j for j, b in enumerate(BUILDINGS) if BUILDING_TYPE[b] == bt[i]]\n            W[i, same if same else slice(None)] = 1.0 / (len(same) if same else len(BUILDINGS))\n    return W\n\n\ndef _soft_onehot(v, k):\n    v = np.asarray(v, dtype=float)\n    out = np.full((len(v), k), 1.0 / k)\n    ok = ~np.isnan(v)\n    out[ok] = 0.0\n    out[np.where(ok)[0], v[ok].astype(int)] = 1.0\n    return out\n\n\nDEFAULT_SPEC = {"hours": "type", "weekend": "building",\n                "slopes": {"previous_usage": "shared", "occupancy": "building",\n                           "temperature": "building", "humidity": "shared"},\n                "temp2": True, "season": True}\n\n\ndef linear_design(c, drop=(), spec=DEFAULT_SPEC):\n    """Design matrix from a clean_input() frame. `drop` = numeric columns unavailable for this\n    missingness-pattern sub-model. Categorical unknowns enter as expectations (soft one-hots)."""\n    if spec.get("missing_spec") and set(drop) & {"occupancy", "previous_usage"}:\n        # sub-models without occupancy/previous_usage must rebuild them from time x building -> richer design\n        spec = {**spec, **spec["missing_spec"]}\n    n = len(c)\n    B = _soft_building(c)\n    T = B @ np.array([[BUILDING_TYPE[b] == t for t in TYPES] for b in BUILDINGS], dtype=float)\n    G = {"building": B, "type": T, "shared": np.ones((n, 1))}\n    K = spec.get("hour_fourier", 0)\n    if K:  # smooth daily profile: K harmonics (missing hour -> 0 = daily average)\n        hh = c["hour"].to_numpy()\n        H = np.concatenate([f(2 * np.pi * k * hh / 24)[:, None] for k in range(1, K + 1) for f in (np.sin, np.cos)], 1)\n        H = np.nan_to_num(H, nan=0.0)\n    else:\n        H = _soft_onehot(c["hour"], 24)\n    dow = c["dow"].to_numpy()\n    we = np.where(np.isnan(dow), 2 / 7, (dow >= 5).astype(float))[:, None]\n    cols = [B, (G[spec["hours"]][:, :, None] * H[:, None, :]).reshape(n, -1), G[spec["weekend"]] * we]\n    if spec.get("hour_dev_scale"):  # per-hour deviations from the smooth profile, shrunk by a heavier\n        # effective ridge penalty (alpha / scale^2) -> captures schedule steps (e.g. residential 08:00 / 21:00)\n        Hd = _soft_onehot(c["hour"], 24) * spec["hour_dev_scale"]\n        Gd = G[spec.get("hour_dev_group", "type")]\n        cols.append((Gd[:, :, None] * Hd[:, None, :]).reshape(n, -1))\n    if spec.get("weekend_hours"):  # separate daily shape on weekends\n        Gw = G[spec["weekend_hours"]]\n        cols.append((Gw[:, :, None] * (H * we)[:, None, :]).reshape(n, -1))\n    for col in NUMERIC:\n        if col not in drop:\n            cols.append(G[spec["slopes"][col]] * c[col].to_numpy()[:, None])\n    # extra shrinkable interactions: (numeric col, group, scale); effective penalty alpha/scale^2\n    if spec.get("interactions"):\n        hr = c["hour"].to_numpy()\n        bands = np.stack([(hr <= 6), (hr >= 7) & (hr <= 11), (hr >= 12) & (hr <= 17), (hr >= 18)], 1).astype(float)\n        bands[np.isnan(hr)] = 0.25\n        we2 = np.concatenate([1 - we, we], 1)\n        GX = {**G, "hourband": bands, "weekend": we2}\n        for col, grp, scale in spec["interactions"]:\n            if col in drop:\n                continue\n            if col == "temp_hinge":\n                v = np.stack([np.maximum(c["temperature"].to_numpy() - k, 0) for k in (26.0, 28.0, 30.0)], 1)\n                if "temperature" in drop:\n                    continue\n                cols.append((GX[grp][:, :, None] * v[:, None, :]).reshape(n, -1) * scale)\n                continue\n            cols.append(GX[grp] * c[col].to_numpy()[:, None] * scale)\n    if spec.get("season", True):\n        m = c["month"].to_numpy()\n        sc = np.stack([np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)], 1)\n        cols.append(np.nan_to_num(sc, nan=0.0))\n    if spec.get("temp2", True) and "temperature" not in drop:\n        cols.append(((c["temperature"].to_numpy() - 28.0) ** 2)[:, None])\n    return np.concatenate(cols, 1)\n\n\ndef pattern_keys(c):\n    miss = c[NUMERIC].isna().to_numpy()\n    return [tuple(col for col, mm in zip(NUMERIC, row) if mm) for row in miss]\n\n\ndef predict_linear(art, df):\n    """Pattern-submodel linear prediction: each row uses the model fitted without its missing\n    numeric columns. art = {"coefs": {pattern: (coef, intercept)}, "spec": ..., "clip": (lo, hi)}."""\n    c = clean_input(df)\n    keys = pattern_keys(c)\n    preds = np.empty(len(c))\n    for k in set(keys):\n        idx = np.array([kk == k for kk in keys])\n        coef, icpt = art["coefs"][k]\n        preds[idx] = linear_design(c[idx], drop=set(k), spec=art["spec"]) @ coef + icpt\n    return np.clip(preds, *art["clip"])\n\n\ndef _dev_keys(F, level):\n    cols = [F[x].cat.codes.to_numpy() if hasattr(F[x], "cat") else F[x].to_numpy() for x in level]\n    return list(zip(*cols))\n\ndef add_dev(F, tables, levels, skip):\n    """Append deviation features (fixed order: level, then column) - identical in training and inference."""\n    G = F.copy()\n    for i, level in enumerate(levels):\n        keys = _dev_keys(F, level)\n        for c in ["occupancy", "previous_usage", "temperature", "humidity"]:\n            if c != skip:\n                mu = np.array([tables[i][c].get(k, np.nan) for k in keys], dtype=float)\n                G[f"dev{i}_{c}"] = F[c].to_numpy(dtype=float) - mu\n    return G\n\nLEVELS = [[\'building\', \'hour\'], [\'building\']]\nNUMS = [\'occupancy\', \'previous_usage\', \'temperature\', \'humidity\']\nV4 = {\'hours\': \'type\', \'weekend\': \'building\', \'slopes\': {\'previous_usage\': \'shared\', \'occupancy\': \'building\', \'temperature\': \'building\', \'humidity\': \'shared\'}, \'temp2\': True, \'season\': True, \'hour_fourier\': 3, \'weekend_hours\': \'shared\', \'missing_spec\': {\'hour_fourier\': 6, \'weekend_hours\': \'type\'}, \'hour_dev_scale\': 0.25}\n\ndef context_design(rows):\n    F=make_features(rows); c=clean_input(rows); n=len(c)\n    B=_soft_building(c)\n    T=B@np.array([[BUILDING_TYPE[b]==t for t in TYPES] for b in BUILDINGS],float)\n    h=F.hour.fillna(12).to_numpy(); we=F.weekend.fillna(2/7).to_numpy()[:,None]\n    H=np.eye(24)[h.astype(int)]\n    phase=2*np.pi*F.month.fillna(6).to_numpy()/12\n    annual=np.column_stack([np.sin(phase),np.cos(phase),np.sin(2*phase),np.cos(2*phase)])\n    daily=np.column_stack([np.sin(2*np.pi*h/24),np.cos(2*np.pi*h/24)])\n    return np.column_stack([B, (T[:,:,None]*H[:,None,:]).reshape(n,-1),\n        .2*(B[:,:,None]*H[:,None,:]).reshape(n,-1), B*we,\n        .5*(T[:,:,None]*(H*we)[:,None,:]).reshape(n,-1), annual,\n        .3*(T[:,:,None]*annual[:,None,:]).reshape(n,-1),\n        .5*(annual[:,:,None]*daily[:,None,:]).reshape(n,-1)])\n\n\ndef profile_values(rows, profiles):\n    D=context_design(rows); F=make_features(rows); out={}\n    for col,p in profiles.items():\n        mu=D@p[\'coef\']+p[\'intercept\']\n        sd=F.building.cat.codes.map(p[\'scales\']).fillna(p[\'default_scale\']).to_numpy(float)\n        out[col]=(mu,sd)\n    return out\n\n\ndef sensor_features(F, tables, skip, profiles=None, values=None, cyclic=False):\n    G=add_dev(F,tables,LEVELS,skip).drop(columns=[skip])\n    if cyclic:\n        for col,period in [(\'hour\',24),(\'month\',12)]:\n            for k in [1,2]:\n                G[f\'{col}_sin{k}\']=np.sin(2*np.pi*k*F[col]/period)\n                G[f\'{col}_cos{k}\']=np.cos(2*np.pi*k*F[col]/period)\n    if profiles is not None:\n        for col in NUMS:\n            mu,sd=values[col]\n            G[\'profile_\'+col]=mu\n            G[\'scale_\'+col]=sd\n            if col!=skip:\n                G[\'profile_dev_\'+col]=(F[col].to_numpy(float)-mu)/sd\n    return G\n\n\ndef predict_sensor(artifact, rows, col, domain=1):\n    F=make_features(rows); F[\'dom\']=domain\n    profiles=artifact[\'profiles\']; values=profile_values(rows,profiles) if profiles else None\n    G=sensor_features(F,artifact[\'tables\'],col,profiles,values,artifact[\'config\'].get(\'cyclic\',False))\n    p=np.mean([lgb.Booster(model_str=s).predict(G,num_threads=4) for s in artifact[\'models\'][col]],axis=0)\n    if artifact[\'config\'].get(\'residual\'): p=values[col][0]+values[col][1]*p\n    return p\n\n\ndef energy_design(frame, config):\n    c = clean_input(frame)\n    spec = dict(V4)\n    spec.update(config.get(\'spec\', {}))\n    D = linear_design(c, spec=spec)\n    B = _soft_building(c)\n    T = B @ np.array([[BUILDING_TYPE[b] == t for t in TYPES] for b in BUILDINGS], float)\n    group = {\'building\': B, \'type\': T, \'shared\': np.ones((len(c), 1))}\n    if config.get(\'steps\'):\n        h = c.hour.to_numpy()\n        residential = T[:, TYPES.index(\'Residential\')]\n        # Generator-like residential switch schedule; Fourier alone smooths the edges.\n        cols = [np.where(np.isfinite(h), ((h >= 8) & (h < 21)).astype(float), 13/24) * residential,\n                np.where(np.isfinite(h), ((h >= 21) | (h < 8)).astype(float), 11/24) * residential]\n        D = np.column_stack([D, *cols])\n    if config.get(\'season_group\'):\n        phase = 2*np.pi*c.month.to_numpy()/12\n        wave = np.nan_to_num(np.column_stack([np.sin(phase), np.cos(phase)]))\n        D = np.column_stack([D, (group[config[\'season_group\']][:,:,None]*wave[:,None,:]).reshape(len(c), -1)])\n    if config.get(\'cooling\'):\n        D = np.column_stack([D, T*np.maximum(c.temperature.to_numpy()[:,None]-24, 0)])\n    return D\n\n\ndef predict_energy(artifact, frame):\n    return energy_design(frame, artifact[\'config\']) @ artifact[\'coef\'] + artifact[\'intercept\']\n\n\ndef fill_sensor_plan(sensors, plan, frame, domain=1, clip_predictions=True):\n    original=frame.reset_index(drop=True)\n    out=original.copy(); clean=clean_input(out)\n    for col in NUMS:\n        raw=clean[col].to_numpy(float); miss=~np.isfinite(raw); out[col]=raw\n        if miss.any():\n            pred=sum(weight*predict_sensor(sensors[name],original.iloc[np.flatnonzero(miss)],col,domain)\n                     for name,weight in plan[col])\n            # Each sensor is predicted from the original observed row, never\n            # from another sensor\'s imputed value.\n            lo,hi=VALID[col]\n            out.loc[np.flatnonzero(miss),col]=np.clip(pred,lo,hi) if clip_predictions else pred\n    return out\n\n\ndef predict_model(artifact,frame):\n    if len(frame)==0: return np.empty(0,dtype=float)\n    with threadpool_limits(limits=4),np.errstate(all=\'ignore\'):\n        filled=fill_sensor_plan(artifact[\'sensors\'],artifact[\'configuration\'][\'sensor_plan\'],frame)\n        p=predict_energy(artifact[\'energy\'],filled)\n        bad=~np.isfinite(p)\n        if bad.any(): p[bad]=predict_linear(artifact[\'fallback\'],frame.iloc[np.flatnonzero(bad)])\n    p=np.clip(np.asarray(p,float),0,400)\n    if p.shape!=(len(frame),) or not np.isfinite(p).all(): raise ValueError(\'Expected one finite prediction per row\')\n    return p\n\n'
_ns = {}
exec(_F007_SRC, _ns)
_p007 = np.asarray(_ns['predict_model'](_both['f007'], test), dtype=float)
artifact = _both
preds = 0.5 * _p11 + 0.5 * _p007  # equal-weight average of the two pipelines
preds = np.asarray(preds, dtype=float).ravel()
assert len(preds) == len(test) and np.isfinite(preds).all()

In [ ]:
# platform input has no id column (template) -> 'prediction' only; with ids -> id,prediction
out = pd.DataFrame({"prediction": preds})
if "id" in test.columns:
    out.insert(0, "id", test["id"].to_numpy())
out.to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(preds)} predictions to {OUTPUT_PATH}")